In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:

!pip install -q fastapi uvicorn pyngrok nest-asyncio
!pip install -q transformers accelerate sentence-transformers

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
)

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device)

print("Models loaded successfully.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Models loaded successfully.


In [4]:
def generate_text(prompt: str, system_prompt: str = "", max_new_tokens: int = 512, temperature: float = 0.3) -> str:
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(llm_model.device)

    with torch.no_grad():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    result = tokenizer.decode(generated, skip_special_tokens=True)
    return result.strip()


In [5]:
def embed_texts(texts: list[str]) -> list[list[float]]:
    embeddings = embed_model.encode(texts, normalize_embeddings=True)
    return embeddings.tolist()

In [6]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Optional
import nest_asyncio
import uvicorn
from pyngrok import ngrok

app = FastAPI(title="Traffic AI Assistant - Kaggle Model Server")


class GenerateRequest(BaseModel):
    prompt: str
    system_prompt: Optional[str] = ""
    max_new_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.3


class GenerateResponse(BaseModel):
    text: str


class EmbedRequest(BaseModel):
    texts: List[str]


class EmbedResponse(BaseModel):
    embeddings: List[List[float]]

In [7]:
@app.get("/health")
def health():
    return {"status": "ok", "device": device}


@app.post("/generate", response_model=GenerateResponse)
def generate(req: GenerateRequest):
    result = generate_text(
        prompt=req.prompt,
        system_prompt=req.system_prompt or "",
        max_new_tokens=req.max_new_tokens,
        temperature=req.temperature,
    )
    return GenerateResponse(text=result)


@app.post("/embed", response_model=EmbedResponse)
def embed(req: EmbedRequest):
    vectors = embed_texts(req.texts)
    return EmbedResponse(embeddings=vectors)


In [ ]:
NGROK_AUTH_TOKEN = "3ENO53BW4aJvHDMkovATGfVNAE8_74TFmRSJrmeii2UJH8FrU"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000).public_url
print("=" * 60)
print(f"Public API URL: {public_url}")
print("=" * 60)

nest_asyncio.apply()

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

await server.serve()

INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public API URL: https://basket-placidly-deftly.ngrok-free.dev
INFO:     197.35.160.126:0 - "GET /health HTTP/1.1" 200 OK
INFO:     197.35.160.126:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.35.160.126:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     197.35.160.126:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.35.160.126:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     197.35.160.126:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.35.160.126:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     197.35.160.126:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.35.160.126:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.35.160.126:0 - "POST /embed HTTP/1.1" 200 OK
INFO:     197.35.160.126:0 - "POST /generate HTTP/1.1" 200 OK
